In [14]:
import osmnx as ox

from genesis.models import Environment

In [2]:
G = ox.load_graphml("tests/data/test_rng.ml")
E = Environment(G)

In [16]:
isinstance(E, Environment)   #=='genesis.models.Environment'

True

In [6]:
import pandas as pd
import numpy as np

d = {1:1, 2:4, 3:6, 4:2}
s = pd.Series(d)

print(max(s))
print(np.mean(s))
print(np.median(s))

6
3.25
3.0


In [5]:
ss=0
for v in d.values():
    ss+=v
print(ss/len(d))

3.25


In [7]:
import this

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!


In [1]:
import logging

In [2]:
logging.debug("A DEBUG Message")
logging.info("An INFO")
logging.warning("A WARNING")
logging.error("An ERROR")
logging.critical("A message of CRITICAL severity")

ERROR:root:An ERROR
CRITICAL:root:A message of CRITICAL severity


In [4]:
logging.basicConfig(level=logging.INFO, filename="py_log.log",filemode="w")
logging.debug("A DEBUG Message")
logging.info("An INFO")
logging.warning("A WARNING")
logging.error("An ERROR")
logging.critical("A message of CRITICAL severity")

ERROR:root:An ERROR
CRITICAL:root:A message of CRITICAL severity


In [2]:
max_time =17.2

print(f"{max_time=}")

max_time=17.2


In [1]:
from genesis.models import Environment
from genesis.calculators import Metrics
from genesis.swiss_knife import ssfpl
import networkx as nx
import osmnx as ox
import logging as lg

def test_calc_ip10():
    E = Environment(ox.load_graphml("tests/data/test_rng.ml"))
    start_node = list(E.G.nodes())[100]
    lngs = nx.single_source_dijkstra_path_length(E.G, start_node, weight='length')
    covered = [1 if t<=10 else 0 for t in lngs.values()]
    ip_real = round(100*sum(covered)/len(covered), 2)

    ip_test = Metrics.calc_metric(E, 
                                   start_node, 
                                   path_function=ssfpl,
                                   metric_function=Metrics.calc_ip,
                                   weight='length')
    lg.basicConfig(level=lg.DEBUG, filename="test_metrics.log", filemode="w")
    # lg.debug(f"{metric_function}: {val=}")
    lg.debug("тестова запись")
    assert ip_real==ip_test

In [6]:
from genesis.calculators import Metrics
from genesis.swiss_knife import ssfpl
from genesis.models import Environment
import numpy as np
import networkx as nx


def create_E():
    G = nx.MultiDiGraph()
    G.add_edge(1, 2, key=0, travel_time=3)
    G.add_edge(1, 3, key=0, travel_time=2)
    G.add_edge(1, 4, key=0, travel_time=5)
    G.add_edge(2, 3, key=0, travel_time=2)
    G.add_edge(2, 5, key=0, travel_time=5)
    G.add_edge(2, 6, key=0, travel_time=7)
    G.add_edge(3, 2, key=0, travel_time=3)
    G.add_edge(3, 7, key=0, travel_time=3)
    G.add_edge(3, 8, key=0, travel_time=4)
    G.add_edge(3, 9, key=0, travel_time=5)
    G.add_edge(4, 3, key=0, travel_time=2)
    G.add_edge(4, 8, key=0, travel_time=4)
    G.add_edge(4, 10, key=0, travel_time=12)
    G.add_edge(4, 11, key=0, travel_time=10)
    G.add_edge(5, 8, key=0, travel_time=8)
    G.add_edge(5, 10, key=0, travel_time=8)
    G.add_edge(5, 12, key=0, travel_time=6)
    G.add_edge(5, 13, key=0, travel_time=9)
    G.add_edge(5, 12, key=0, travel_time=8)
    G.add_edge(6, 8, key=0, travel_time=1)
    G.add_edge(6, 9, key=0, travel_time=4)
    G.add_edge(6, 12, key=0, travel_time=4)
    G.add_edge(7, 10, key=0, travel_time=4)
    G.add_edge(7, 11, key=0, travel_time=9)
    G.add_edge(7, 13, key=0, travel_time=7)
    G.add_edge(8, 10, key=0, travel_time=6)
    G.add_edge(8, 13, key=0, travel_time=8)
    G.add_edge(9, 14, key=0, travel_time=10)
    G.add_edge(10, 13, key=0, travel_time=7)
    G.add_edge(10, 14, key=0, travel_time=5)
    G.add_edge(11, 12, key=0, travel_time=7)
    G.add_edge(11, 15, key=0, travel_time=7)
    G.add_edge(12, 13, key=0, travel_time=8)
    return Environment(G)

E = create_E()
print("max:", Metrics.calc_metric(E, 1, ssfpl, np.max))
print("mean:", Metrics.calc_metric(E, 1, ssfpl, np.mean))
print("median:", Metrics.calc_metric(E, 1, ssfpl, np.median))
print("ip-10:", Metrics.calc_metric(E, 1, ssfpl, Metrics.calc_ip()))
print("ip-20:", Metrics.calc_metric(E, 1, ssfpl, Metrics.calc_ip(ip_val=20)))

max: 21
mean: 8.67
median: 8.0
ip-10: 66.67
ip-20: 93.33


In [1]:
import osmnx as ox
import networkx as nx
import numpy as np
import pandas as pd

from genesis.models import Environment
from genesis.calculators import Metrics
from genesis.swiss_knife import ssfpl

E = Environment(ox.load_graphml("tests/data/test_rng.ml"))

In [6]:
def calc_metric(E:Environment, 
                    node:int, 
                    # path_function, 
                    # metric_function, 
                    # weight: str = "travel_time",
                    # precision: int = 2,
                    **kwargs):
        route_lens = nx.single_source_dijkstra_path_length(E.G, node, weight='length')
        less_500 = [1 if d<500 else 0 for d in route_lens.values()]
        return route_lens, sum(less_500)/len(less_500)

start_node = list(E.G.nodes())[2000]
route_lens, cm = calc_metric(E, start_node)
# assert calc_metric(E, start_node)==100

In [1]:
from genesis.models import SpeedProfile

sp = SpeedProfile()
sp.kmh_to_mm_precision=0
# sp.set_speeds_5([50,40,30,20,10])
# assert sp.sp["corridor"]==167
sp._sp

{'motorway': 40,
 'motorway_link': 40,
 'trunk': 40,
 'trunk_link': 40,
 'primary': 40,
 'primary_link': 40,
 'secondary': 40,
 'secondary_link': 40,
 'tertiary': 30,
 'tertiary_link': 30,
 'road': 30,
 'unclassified': 30,
 'service': 25,
 'residential': 25,
 'living_street': 25,
 'track': 25,
 'footway': 10,
 'path': 10,
 'pedestrian': 10,
 'steps': 5,
 'cycleway': 5,
 'bridleway': 5,
 'corridor': 5}

In [2]:
sp.__dict__

{'_sp': {'motorway': 40,
  'motorway_link': 40,
  'trunk': 40,
  'trunk_link': 40,
  'primary': 40,
  'primary_link': 40,
  'secondary': 40,
  'secondary_link': 40,
  'tertiary': 30,
  'tertiary_link': 30,
  'road': 30,
  'unclassified': 30,
  'service': 25,
  'residential': 25,
  'living_street': 25,
  'track': 25,
  'footway': 10,
  'path': 10,
  'pedestrian': 10,
  'steps': 5,
  'cycleway': 5,
  'bridleway': 5,
  'corridor': 5},
 'kmh_to_mm_precision': 0}

# Тест использования интерфейсов

In [7]:
import osmnx as ox
from genesis.models import Environment
from genesis.interfaces import ISpatialFeature
import networkx as nx

In [ ]:
G = ox.load_graphml("tests/data/test_rng.ml")

In [2]:
def tr(G, poly):
    return ox.truncate.truncate_graph_polygon(G, poly)

E = Environment()

G.truncate = tr
E.add_spatial_feature(G)

TypeError: аргумент 'spatial_feature' должен реализовывать интерфейс 'ISpatialFeature'!

In [9]:
class GDS(nx.MultiDiGraph, ISpatialFeature):
    def __init__(self, incoming_graph_data=None, multigraph_input=None, **attr):
        super().__init__(incoming_graph_data, multigraph_input, **attr)
    
    def truncate(self, area):
        return ox.truncate.truncate_graph_polygon(self, area)

In [10]:
gds = GDS(G)


In [13]:
print(isinstance(gds, nx.MultiDiGraph))
print(isinstance(gds, ISpatialFeature))

True
True


In [17]:
class GDS(nx.MultiDiGraph, ISpatialFeature):
    def __init__(self, incoming_graph_data=None, multigraph_input=None, **attr):
        super().__init__(incoming_graph_data, multigraph_input, **attr)

    def truncate(self, area):
        return ox.truncate.truncate_graph_polygon(self, area)

In [22]:
gds = GDS(G)
gds.name = 'G'

In [24]:
gds.name

''

In [34]:
from shapely.geometry import MultiPolygon, Polygon


class RoadNetworkGraph(nx.MultiDiGraph, ISpatialFeature):
    def __init__(self, incoming_graph_data=None, multigraph_input=None, name='RNG', **attr):
        super().__init__(incoming_graph_data, multigraph_input, name=name, **attr)

    def truncate(self, polygon: Polygon, **attr):
        return ox.truncate.truncate_graph_polygon(self, polygon, **attr)
    
gds = RoadNetworkGraph(G)
gds.name

'RNG'

In [35]:
str(gds)

"RoadNetworkGraph named 'RNG' with 5514 nodes and 12199 edges"

In [7]:
from genesis.models import RoadNetworkGraph
from genesis.interfaces import IEnvironment, ISpatialFeature
import osmnx as ox

G = ox.load_graphml("tests/data/test_rng.ml")
G = RoadNetworkGraph(G)

In [29]:
class Environment2(IEnvironment):        # Это уже реализация!!!
    '''
    Базовая реализация модели окружения. 
    В нее входит ГДС, размещение подразделений, объектов, и т.д.
    '''
    def __init__(self):
        # self.spatial_data = {}
        self.spatial_data = object()

    def add_spatial_feature(self, spatial_feature: ISpatialFeature):
        '''Добавка пространственных данных'''
        # Нужно проверить наследует ли spatial_feature интерфейсу ISpatialFeature
        if not isinstance(spatial_feature, ISpatialFeature):
            raise TypeError("аргумент 'spatial_feature' должен \
                            реализовывать интерфейс 'ISpatialFeature'!")

        # self.spatial_data[spatial_feature.name] = spatial_feature
        # self.spatial_data.__dict__[spatial_feature.name] = spatial_feature
        setattr(self.spatial_data , spatial_feature.name, spatial_feature)
        return self

    def test(self):
        pass

E = Environment2()
E.add_spatial_feature(G)

AttributeError: 'object' object has no attribute '__dict__'

In [28]:
E.RNG.number_of_nodes()

5514

In [26]:
o = object
setattr(o, 't', 20)
o

TypeError: can't set attributes of built-in/extension type 'object'

In [33]:
import pandas as pd

df = pd.DataFrame(name='123')
df.__name__

TypeError: __init__() got an unexpected keyword argument 'name'

In [ ]:
class dff(pd.DataFrame):
    def __name__